In [ ]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

**The first step of this data analysis pipeline is EDA. It consists of the following main steps:**
* Summarize dataset size and structure. 
* Provide descriptive statistics for numerical features. 
* Show distributions and visualizations of key variables. 
* Check for missing values, outliers, or anomalies. 
* Explore correlations or relationships among features. 
* Check for class balance.




In [ ]:
# Load the dataset
df = pd.read_csv('../data/train.csv')
# Show the first few rows of the dataset
df.head()

,AnimalID,Name,DateTime,OutcomeType,OutcomeSubtype,AnimalType,SexuponOutcome,AgeuponOutcome,Breed,Color
0,A671945,Hambone,2014-02-12 18:22:00,Return_to_owner,NaN,Dog,Neutered Male,1 year,Shetland Sheepdog Mix,Brown/White
1,A656520,Emily,2013-10-13 12:44:00,Euthanasia,Suffering,Cat,Spayed Female,1 year,Domestic Shorthair Mix,Cream Tabby
2,A686464,Pearce,2015-01-31 12:28:00,Adoption,Foster,Dog,Neutered Male,2 years,Pit Bull Mix,Blue/White
3,A683430,NaN,2014-07-11 19:09:00,Transfer,Partner,Cat,Intact Male,3 weeks,Domestic Shorthair Mix,Blue Cream
4,A667013,NaN,2013-11-15 12:52:00,Transfer,Partner,Dog,Neutered Male,2 years,Lhasa Apso/Miniature Poodle,Tan


In [ ]:
# Dataset information
df.info()

**Dataset**

The Shelter Animal Outcomes dataset represents a socially meaningful and technically manageable classification problem.
The training file contains about 26 000 rows × 9 columns. Most variables are categorical or date/time text, requiring straightforward preprocessing with pandas and scikit-learn. The dataset is well-structured and free of excessive missing data, making it practical for an introductory analytics pipeline.

| **Feature Name** | **Type** | **Units (if applicable)** | Description  |
|:-----------------|:---------|:--------------------------|:------------------------------------|
| **OutcomeType** | Categorical  | – | Final outcome for each animal: Adoption, Died, Euthanasia, Return_to_owner, or Transfer. |
| OutcomeSubtype | Categorical | – | More detailed reason for the outcome |
| AnimalID | Identifier | – | Unique ID assigned to each animal upon intake; not used as a model feature. |
| Name | Categorical  | – | Animal’s given name . Missing values mean no name.  |
| DateTime | Datetime (string) | – | Date and time the outcome was recorded. Can be split into year, month, day, hour, weekday. |
| AnimalType | Categorical | – | Type of animal . Used to analyse species differences in outcomes. |
| SexuponOutcome | Categorical | – | Combined sex and sterilisation status. Can be split into two features: Sex and Neutered. |
| AgeuponOutcome | Numerical | **years** (after conversion) | Age of the animal at outcome time converted to years. |
| Breed | Categorical (Text) | – | Breed description; may include “Mix.”  |
| Color | Categorical (Text) | – | Colours of the animal. Can be split into Primary and Secondary Colour. |



In [ ]:
# Provide descriptive statistics for numerical features
# Converts AgeUponOutcome, which can be in weeks, months and years, into age_years
#which is all years
def convert_age_to_years(age_str):
    if pd.isna(age_str):
        return None

    age_str = age_str.lower().strip()
    parts = age_str.split()
    if len(parts) != 2:
        return None  

    num, unit = parts
    try:
        num = float(num)
    except ValueError:
        return None

    if "day" in unit:
        return num / 365
    elif "week" in unit:
        return num / 52
    elif "month" in unit:
        return num / 12
    elif "year" in unit:
        return num
    else:
        return None

df['age_years'] = df['AgeuponOutcome'].apply(convert_age_to_years)

In [ ]:
# Summary statistics of age
df['age_years'].describe()

In [ ]:
# Boxplot of age
df.boxplot('age_years', vert=False)
plt.xlabel("Age in Years")
plt.title("Boxplot of Age in Years of Animals")
plt.show()
# We can see that the average age of animals is approximiately 2 years, 
# and the distribution is right skewed

In [ ]:
#filter dataset into cats and dogs
cats = df[df['AnimalType'] == 'Cat']
dogs = df[df['AnimalType'] == 'Dog']

In [ ]:
#Show distribution and visualization of key variables
#Bar chart of cats vs dogs
df['AnimalType'].value_counts().plot(kind='bar')
plt.ylabel("Count")
plt.title("Animal Distribution")
plt.show()

In [ ]:
# Summary statistics for cats
cats['age_years'].describe()

In [ ]:
# Summary statistics for dogs
dogs['age_years'].describe()

In [ ]:
# Boxplot of age for cats
cats.boxplot('age_years', vert=False)
plt.xlabel("Age in Years")
plt.title("Boxplot of Age in Years of Cats")
plt.show()

In [ ]:
# Boxplot of age for dogs
dogs.boxplot('age_years', vert=False)
plt.xlabel("Age in Years")
plt.title("Boxplot of Age in Years of Dogs")
plt.show()
#We can see that dogs live longer on average compared to cats

In [ ]:
#Sex upon outcome of all animals
df['SexuponOutcome'].value_counts().plot(kind='bar')
plt.ylabel("Count")
plt.title("Sex upon outcome of animals")
plt.show()

In [ ]:
#Sex upon outcome of animals, split into cats and dogs
animal_gender = pd.crosstab(df['SexuponOutcome'], df['AnimalType'])
animal_gender.plot(kind='bar')
plt.ylabel("Count")
plt.title("Sex upon outcome of animals")
plt.show()
#We can see dogs are left intact less often than cats

In [ ]:
#Outcomes of all animals
df['OutcomeType'].value_counts().plot(kind='bar')
plt.ylabel("Count")
plt.title("Outcome type of animals")
plt.show()

In [ ]:
#Outcome of animals split into dogs and cats
outcome_animals = pd.crosstab(df['OutcomeType'], df['AnimalType'])
outcome_animals.plot(kind='bar')
plt.ylabel("Count")
plt.title("Outcome Type of Animals")
plt.show()

In [ ]:
# Checking for missing values, outliers, or anomalies

# Method 'isnull()' returns True if the data is missing, False otherwhise 
missing_data = df.isnull()

In [ ]:
# For loop to figure out the number of missing values from each column
for column in missing_data.columns.values.tolist():
    print(column)
    print (missing_data[column].value_counts())
    print("") 



**4 of the columns contain missing data. Considering the dataset contains 26729 rows of data:**
* 'Name': 7691 missing data
* 'OutcomeSubtype': 13117 missing data
* 'SexuponOutcome': 1 missing data
* 'AgeuponOutcome': 18 missing data

In [ ]:
# === Handling Missing Values ===

# 'Name':
# We will have a column of ints where 1 means the animal has a name, and 0 means it does not.
df['HasName'] = (df['Name'].fillna('') != '').astype(int)

# 'OutcomeSubtype':
# In principle, this variable could serve as an additional predictive target.
# However, due to the presence of nearly 50% missing entries, imputation is not appropriate
# and its inclusion would compromise data reliability.
# The feature is therefore excluded from further analysis to ensure consistent and interpretable results.
if 'OutcomeSubtype' in df.columns:
    df.drop(columns=['OutcomeSubtype'], inplace=True)

# 'SexuponOutcome':
# Only one value is missing in this feature.
# To balance functionality and statistical robustness, we replace the missing entry with the most frequent category (mode imputation).
df['SexuponOutcome'] = df['SexuponOutcome'].fillna(df['SexuponOutcome'].mode()[0])

# 'AgeuponOutcome':
# For this numerical feature, we'll apply a multivariate imputation using a K-Nearest Neighbors (KNN) approach.
# KNNImputer leverages correlated features to infer realistic values for missing ages, ensuring consistency.
# Normalization or standardization is required.

# select potential related features, then create a copy to not mess with any original data
features_for_imputation = ['AnimalType', 'SexuponOutcome', 'age_years']
impute_df = df[features_for_imputation].copy()

# encode catagorical features to numerical using one hot encoding
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded = encoder.fit_transform(impute_df[['AnimalType', 'SexuponOutcome']])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['AnimalType', 'SexuponOutcome']))

# combine all into one
combined = pd.concat([impute_df[['age_years']].reset_index(drop=True), encoded_df.reset_index(drop=True)], axis=1)

#standardize
scaler = StandardScaler()
scaled = scaler.fit_transform(combined)

# apply KNN with 3 as k
imputer = KNNImputer(n_neighbors=3)
imputed_scaled = imputer.fit_transform(scaled)

#reverse scale
imputed = scaler.inverse_transform(imputed_scaled)

# apply to data set
df['age_years'] = imputed[:, 0]